# Attention: SDPA, GQA, FlashAttention — Math, Derivation & CuTeDSL

## 1. Where Attention Sits in Inference

Attention is the unique **O(T²)** operation in the transformer. Every other sublayer (MLP, LayerNorm, projections) is O(T) in sequence length — only attention scales quadratically.

```
Q_rope ∈ ℝ^{B×T×32×128}   ← from RoPE
K_rope ∈ ℝ^{B×T×8×128}    ← from RoPE
V      ∈ ℝ^{B×T×8×128}    ← from V projection (no RoPE)
  │
  │  ① Scores:  S = Q_rope @ K_rope.T / sqrt(128)   [B×32×T×T]  ← O(T²) !
  │  ② Mask:    S[t,s] = -∞ for s > t (causal)
  │  ③ Weights: P = softmax(S, dim=-1)              [B×32×T×T]
  │  ④ Output:  O = P @ V                           [B×32×T×128]
  │
  └→ reshape O → [B×T×4096] → O projection
```

**The problem FlashAttention solves:** At T=4096, step ① creates a 32×4096×4096 matrix **per batch element** in FP32:

$$32 \times 4096 \times 4096 \times 4 \text{ bytes} = 2 \text{ GB}$$

This doesn't fit in L2 cache (6 MB on RTX 4080) or SMEM (128 KB/SM). Without FlashAttention, the score matrix is allocated in HBM and read/written multiple times — making attention memory-bound by a large factor.

## 2. What Are Q, K, V?

### Conceptually

Consider the word "bank" in the sentence "I sat by the river bank." It should attend to "river" more than "money." Attention achieves this by having each token produce three vectors after the projection GEMMs:

- **Q (Query):** "What context am I searching for?" — what token t wants to know about the other tokens.
- **K (Key):** "What do I advertise about myself?" — what token s says it's about.
- **V (Value):** "What do I actually contribute?" — the information token s donates if attended to.

The dot product Q·K measures how well token t's query matches token s's key. High dot product → high attention weight → more of token s's value flows into token t's output.

### Mathematically

For each attention head h, Q, K, V are projections of the normalized hidden state X:

$$Q^{(h)} = X W_Q^{(h)} \in \mathbb{R}^{T \times D_h}$$
$$K^{(h)} = X W_K^{(h)} \in \mathbb{R}^{T \times D_h}$$
$$V^{(h)} = X W_V^{(h)} \in \mathbb{R}^{T \times D_h}$$

where $X \in \mathbb{R}^{T \times D}$ is the normalized hidden state, $W_Q^{(h)} \in \mathbb{R}^{D \times D_h}$ is the Q-head projection weight.

In Qwen3-8B: D=4096, D_h=128. Q has H_q=32 heads, K and V have H_kv=8 heads (GQA).

### GQA (Grouped Query Attention)

With H_q=32 Q-heads and H_kv=8 KV-heads, each K head is shared by group_size=4 Q-heads:

$$\text{head group } g = \{4g,\; 4g+1,\; 4g+2,\; 4g+3\}$$
$$\text{all 4 Q-heads in group } g \text{ attend to the same } K^{(g)},\; V^{(g)}$$

This cuts KV memory and bandwidth by **4×** compared to Multi-Head Attention (MHA).

### KV Cache

At decode step t, only the new token has a new Q. K and V for positions 0..t−1 are reused from cache:

- **Cache K:** $K_\text{cache} \in \mathbb{R}^{B \times T \times H_{kv} \times D_h}$ (grows by 1 row each decode step)
- **Cache V:** same shape
- **Bandwidth to read:** $B \times T \times H_{kv} \times D_h \times 2 \times 2$ bytes (K and V both BF16)
  $$= 2 \times T \times 8 \times 128 \times 2 = 4096 \times T \text{ bytes per layer}$$

## 3. The SDPA Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{D_h}}\right) V$$

**Element-wise** for one query token t and one head h:

**Step 1 — Scores** (dot product of query t with every key s):
$$\text{score}_{t,s} = \frac{\displaystyle\sum_{d=0}^{D_h-1} Q^{(h)}_{t,d} \cdot K^{(h)}_{s,d}}{\sqrt{D_h}}$$

**Step 2 — Weights** (numerically stable softmax over scores):
$$p_{t,s} = \frac{\exp\!\left(\text{score}_{t,s} - \max_{s'}\text{score}_{t,s'}\right)}{\displaystyle\sum_{s''} \exp\!\left(\text{score}_{t,s''} - \max_{s'}\text{score}_{t,s'}\right)}$$

**Step 3 — Output** (weighted sum of values, causal: sum only over s ≤ t):
$$O^{(h)}_{t,d} = \sum_{s=0}^{t} p_{t,s} \cdot V^{(h)}_{s,d}$$

The upper limit t in the sum implements the **causal mask**: token t can only attend to positions ≤ t (no peeking at future tokens). Positions s > t get score −∞, which maps to weight 0 after softmax.

## 4. Derivation: Why Divide by sqrt(D_h)?

Full derivation from statistics.

### Without scaling

If Q and K elements are i.i.d. with mean 0, variance 1, consider the dot product:

$$Q_t \cdot K_s = \sum_{d=0}^{D_h-1} Q_{t,d} \, K_{s,d}$$

Each term $Q_{t,d} K_{s,d}$ has mean 0 and variance 1 (product of two independent zero-mean unit-variance variables). The **sum of D_h independent terms** has variance D_h, so:

$$\text{Std}(Q_t \cdot K_s) = \sqrt{D_h} = \sqrt{128} \approx 11.3$$

Raw dot products range roughly ±11. Feed these into softmax:

$$\text{softmax}([12,\; 4,\; -4]) \approx [0.9997,\; 0.0003,\; 0.0000]$$

The distribution **collapses to a near-delta function**. The gradient of softmax is ~0 near these extremes → training stalls (vanishing gradients). At inference, a collapsed attention that attends to only one position produces worse output than one that blends across multiple positions.

### With scaling by 1/sqrt(D_h)

Dividing by $\sqrt{D_h}$ rescales the scores to have std dev ≈ 1:

$$\text{softmax}\!\left(\frac{[12, 4, -4]}{\sqrt{128}}\right) = \text{softmax}([1.06,\; 0.35,\; -0.35]) \approx [0.48,\; 0.31,\; 0.21]$$

The softmax is now in a **healthy regime**: outputs are spread, gradients are nonzero, and the model can learn to attend to multiple positions simultaneously.

### Why sqrt specifically?

Because for independent random variables, $\text{Var}\!\left(\sum_d X_d Y_d\right) = D_h \cdot \text{Var}(X)\text{Var}(Y)$. Dividing the sum by $\sqrt{D_h}$ normalizes the variance back to $\text{Var}(X)\text{Var}(Y)=1$. This is the same reasoning behind He initialization ($1/\sqrt{n_\text{in}}$) and layer normalization.

## 5. FlashAttention Derivation: Online Softmax

The key insight: we don't need to store the full T×T score matrix to compute softmax(S)V.

### Online softmax algorithm

Process keys and values in tiles of size B_c. For each query t, maintain a **running state of three scalars**:
- **m**: running maximum score (initialized to −∞)
- **d**: running denominator of softmax (initialized to 0)
- **acc**: running weighted sum of values (initialized to **0** vector of size D_h)

When we see a new score $s_\text{new}$ and corresponding value $v_\text{new}$:

$$m_\text{new} = \max(m,\; s_\text{new})$$

$$d_\text{new} = d \cdot e^{m - m_\text{new}} + e^{s_\text{new} - m_\text{new}}$$

$$\text{acc}_\text{new} = \text{acc} \cdot e^{m - m_\text{new}} + e^{s_\text{new} - m_\text{new}} \cdot v_\text{new}$$

$$m \leftarrow m_\text{new}, \qquad d \leftarrow d_\text{new}, \qquad \text{acc} \leftarrow \text{acc}_\text{new}$$

**Final output:** $\text{acc} \,/\, d$

### Why the rescaling factor works

The rescaling factor $e^{m - m_\text{new}}$ corrects the old running state when a **larger maximum is found**. It multiplies the previously-accumulated numerator and denominator by the same factor, which is equivalent to subtracting the new (larger) maximum from all previously-seen scores — exactly what numerically stable softmax requires.

When no new maximum appears, $e^{m - m_\text{new}} = e^0 = 1$ — the correction is free (no-op multiply).

### Key result

The three scalars (m, d, acc) **live in registers**. The T×T score matrix never needs to exist in memory. This is the algorithmic core of FlashAttention.

**FLOP count with FlashAttention** (prefill, causal):
- Score: $2 \cdot H_q \cdot T^2 \cdot D_h$ FLOPs
- PV:    $2 \cdot H_q \cdot T^2 \cdot D_h$ FLOPs
- Total: $4 \cdot H_q \cdot T^2 \cdot D_h$

**Bytes read** (Q, K, V in BF16, no T×T materialization):
- $2 \cdot (H_q + 2 H_{kv}) \cdot T \cdot D_h \cdot 2$ bytes

**Arithmetic intensity:** $\sim T/4$ FLOP/byte — compute-bound above T ≈ 600 on RTX 4080 (ridge ≈ 151).

In [ ]:
import torch, math

torch.manual_seed(0)
T, D_h = 8, 4   # small example

Q = torch.randn(T, D_h)
K = torch.randn(T, D_h)
V = torch.randn(T, D_h)
scale = 1.0 / math.sqrt(D_h)

# ── Level 0: high-level intent ───────────────────────────────────
# O = attention(Q, K, V)

# ── Level 1: expand the formula ─────────────────────────────────
# O = softmax(Q @ K.T / sqrt(D_h)) @ V

# ── Level 2: expand softmax ─────────────────────────────────────
# scores = Q @ K.T / sqrt(D_h)      [T, T]
# weights = exp(scores - max(scores)) / sum(exp(scores - max(scores)))
# O = weights @ V

# ── Level 3: add causal mask ─────────────────────────────────────
# scores[t, s] = -inf for s > t

# ── Level 4: fully explicit ──────────────────────────────────────
scores = Q @ K.T * scale                              # [T, T]

mask   = torch.triu(torch.ones(T, T), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))      # causal mask

scores_max  = scores.max(dim=-1, keepdim=True).values
exp_scores  = torch.exp(scores - scores_max)          # numerically stable
exp_scores  = exp_scores.masked_fill(mask, 0.0)       # zero out masked
weights     = exp_scores / exp_scores.sum(dim=-1, keepdim=True)   # [T, T]

O = weights @ V   # [T, D_h]

# ── Reference check ────────────────────────────────────────────────
O_ref = torch.nn.functional.scaled_dot_product_attention(
    Q.unsqueeze(0).unsqueeze(0),
    K.unsqueeze(0).unsqueeze(0),
    V.unsqueeze(0).unsqueeze(0),
    is_causal=True,
    scale=scale
).squeeze(0).squeeze(0)

print(f"Q shape: {list(Q.shape)}, K shape: {list(K.shape)}, V shape: {list(V.shape)}")
print(f"scores shape: {list(scores.shape)}  (the T×T matrix)")
print(f"weights shape: {list(weights.shape)}")
print(f"O shape: {list(O.shape)}")
print(f"Max diff vs torch SDPA: {(O - O_ref).abs().max():.2e}")
assert (O - O_ref).abs().max() < 1e-4
print("✓ Level 4 matches torch.nn.functional.scaled_dot_product_attention")

In [ ]:
# Implement the online softmax that FlashAttention uses.
# No T×T matrix ever exists — just 3 running scalars per query.

def online_softmax_attention(Q, K, V, scale, B_c=2):
    """
    FlashAttention core: process KV in tiles of B_c.
    Returns the same output as standard attention.
    """
    T, D_h = Q.shape
    output = torch.zeros_like(Q)

    for t in range(T):   # for each query token
        q_t = Q[t]       # [D_h] — this query stays in "registers" the whole time

        m_running   = float('-inf')   # running max score
        d_running   = 0.0             # running denominator
        acc         = torch.zeros(D_h)   # running output accumulator

        # Process KV in tiles (at t, causal: only positions 0..t)
        for s_start in range(0, t+1, B_c):
            s_end = min(s_start + B_c, t+1)

            for s in range(s_start, s_end):
                # Score: dot product of q_t and k_s
                score = (q_t * K[s]).sum().item() * scale

                # Online update
                m_new = max(m_running, score)
                rescale = math.exp(m_running - m_new) if m_running != float('-inf') else 0.0
                d_running = d_running * rescale + math.exp(score - m_new)
                acc       = acc       * rescale + math.exp(score - m_new) * V[s]
                m_running = m_new

        output[t] = acc / d_running

    return output

O_flash = online_softmax_attention(Q, K, V, scale, B_c=2)
print(f"Online softmax vs standard: max diff = {(O_flash - O).abs().max():.2e}")
assert (O_flash - O).abs().max() < 1e-4
print("✓ Online softmax (FlashAttention core) matches standard attention")
print()
print("State per query token: m (1 scalar) + d (1 scalar) + acc (D_h scalars) = ", 1+1+D_h, "values")
print("T×T score matrix needed: NO — each score is computed and immediately discarded")

## 6. Symbol Table

| Symbol | Definition | Python / PyTorch | CuTeDSL / CUDA | Triton |
|---|---|---|---|---|
| Q | Query tensor (after RoPE) | `q: Tensor[B,H_q,T,D_h]` | `mQ: cute.Tensor` | `Q_ptr` |
| K | Key tensor (from KV cache) | `k: Tensor[B,H_kv,T,D_h]` | `mK: cute.Tensor` | `K_ptr` |
| V | Value tensor (from KV cache) | `v: Tensor[B,H_kv,T,D_h]` | `mV: cute.Tensor` | `V_ptr` |
| D_h | Head dimension | `head_dim = 128` | `D = 128` | `HEAD_DIM: tl.constexpr` |
| H_q | Number of Q heads | `32` | `H_Q = 32` | |
| H_kv | Number of KV heads | `8` | `H_KV = 8` | |
| group_size | Q heads per KV head | `4` | `GROUP = 4` | |
| scale | 1/sqrt(D_h) | `1 / math.sqrt(128)` | `scale` | `SCALE: tl.constexpr` |
| S | Score matrix QK^T/sqrt(D_h) | `scores = Q @ K.T * scale` | computed tile by tile | computed tile by tile |
| m | Running max (online softmax) | `m = float('-inf')` | `m_running` | `m_i` |
| d | Running denominator | `d = 0.0` | `d_running` | `l_i` |
| acc | Running output accumulator | `acc = zeros(D_h)` | `rO` (register) | `acc` |
| B_c | KV tile size (FlashAttention) | `64` (typical) | `Bc = 64` | `BLOCK_N: tl.constexpr` |
| B_r | Query tile size (FA-2) | `64` (typical) | `Br = 64` | `BLOCK_M: tl.constexpr` |

## 7. GQA: Grouped Query Attention

With H_q=32 and H_kv=8 (group_size=4), map Q-head h to KV-head h//4:

$$\text{score}^{(h)}_{t,s} = \frac{Q^{(h)}_t \cdot K^{(\lfloor h/4 \rfloor)}_s}{\sqrt{D_h}}$$

Q-heads 0,1,2,3 all attend using K[0] and V[0]; heads 4,5,6,7 use K[1] and V[1]; and so on.

### Memory savings at T=4096

| Config | KV heads | KV cache per layer | 36 layers total |
|---|---|---|---|
| MHA (H_kv=32) | 32 | 2 × 32 × 4096 × 128 × 2 = **64 MB** | 36 × 64 = 2,304 MB |
| GQA (H_kv=8)  |  8 | 2 × 8 × 4096 × 128 × 2 = **16 MB** | 36 × 16 = **576 MB** |

GQA requires **4× less KV memory and bandwidth** than MHA — the key enabler for long-context inference on consumer GPUs.

### Bandwidth per decode step

Reading the full KV cache for 1 new token (per layer):

$$2 \times H_{kv} \times T \times D_h \times 2 \text{ bytes} = 4096 \times T \text{ bytes}$$

At T=4096, 36 layers:
$$4096 \times 4096 \times 36 = 576 \text{ MB of KV reads per decode step}$$

At 380 GB/s bandwidth: ~1.5 ms per decode step just for attention KV reads (before compute). This is why batching matters: with B=32 users, those 576 MB of reads are amortized across 32 output tokens generated simultaneously.

## 8. Data Flow at Scale: Multi-User KV Cache

### Continuous batching and PagedAttention

With B=32 concurrent users, each at a different context length T_i:
- User 0 might be at T=100, user 1 at T=4096, etc.
- Naive padding to max T wastes KV memory on short users.
- **PagedAttention** (vLLM/SGLang): KV cache is divided into fixed-size blocks (e.g., block_size=16 tokens). Each user gets blocks allocated dynamically; no padding needed.

At B=32 users with avg T=2000:
- KV memory: 32 × 2000 tokens × 36 layers × 4 KB/token/layer = **9.2 GB** — fits on a 16 GB RTX 4080
- Bandwidth per decode step: 32 × 2000 × 4 KB × 36 = **9.2 GB of KV reads**
- At 380 GB/s: ~24 ms just for KV reads — amortized across 32 simultaneous users

### Flash-Decoding at long context

**What problem does Flash-Decoding solve?**

With standard FlashAttention, one GPU thread-block handles one (user, head) pair. For B=1 user with 32 Q-heads, that's only 32 thread-blocks — but the RTX 4080 has 76 SMs. Most SMs sit idle. As context T grows, each thread-block does more work, but you still only have 32 of them running at once.

**The fix:** split the KV sequence into P chunks and process them in parallel. Instead of 1 thread-block per head looping over all T KV tokens, use P thread-blocks per head — each processing T/P tokens — then merge their results. Now you have P×32 blocks running simultaneously, keeping all SMs busy.

**How the merge works:** Each of the P blocks produces a partial online softmax state (m_p, d_p, acc_p). These are merged using the same rescaling rule as the online softmax itself:

$$m_\text{final} = \max_p(m_p)$$

$$d_\text{final} = \sum_p d_p \cdot e^{m_p - m_\text{final}}$$

$$\text{acc}_\text{final} = \frac{\displaystyle\sum_p \text{acc}_p \cdot e^{m_p - m_\text{final}}}{d_\text{final}}$$

The merger writes and reads P partial state tensors — a small overhead that pays off when T≥2048 and batch is small. FlashInfer uses Flash-Decoding internally for `single_decode_with_kv_cache` when context length exceeds a threshold.

In [ ]:
# Show how the online softmax state maps to CuTeDSL register layout.
# This is pseudocode corresponding to sm89_v2_flash_tiled_kv.

# Key ideas:
# 1. Q stays in registers throughout (loaded once per CTA)
# 2. K and V tiles are loaded into SMEM via cp.async (double-buffered)
# 3. The (m, d, acc) state is in registers — never written to HBM
# 4. At the end, only the output O is written to HBM

print("FlashAttention CuTeDSL register layout:")
print()
print("  Registers per thread (D_h=128, BLOCK=128 threads):")
print("    rQ:    D_h / BLOCK = 1 float per thread (query fragment)")
print("    rK:    B_c × (D_h/BLOCK) floats (key fragment in tile)")
print("    rV:    B_c × (D_h/BLOCK) floats (value fragment in tile)")
print("    m:     1 float (running max)")
print("    d:     1 float (running denominator)")
print("    rO:    D_h / BLOCK floats (output accumulator)")
print()
print("  SMEM per CTA:")
print("    sK:    B_c × D_h × 2 bytes = 64 × 128 × 2 = 16 KB  (K tile)")
print("    sV:    B_c × D_h × 2 bytes = 64 × 128 × 2 = 16 KB  (V tile)")
print("    Total: 32 KB (single buffer) → 64 KB (double buffered)")
print()
print("  The T×T score matrix: NOT ALLOCATED — each score computed, used, discarded")
print()
print("  KV tile loop structure:")
print("    for kv_start in range(0, T, B_c):")
print("      if kv_start > q_t: continue  # causal: skip future tiles")
print("      async_load K_tile → sK       # cp.async fires in background")
print("      async_load V_tile → sV")
print("      pipeline_wait()")
print("      for s in range(B_c):")
print("        score = dot(q, sK[s]) * scale")
print("        # online update: m, d, acc")
print("        m_new = max(m, score)")
print("        d = d * exp(m-m_new) + exp(score-m_new)")
print("        acc = acc * exp(m-m_new) + exp(score-m_new) * sV[s]")
print("        m = m_new")
print("    output = acc / d")

## 9. Production Attention Kernels

| Library | Kernel | What it solves | Fusion |
|---|---|---|---|
| FlashAttention-2 | `flash_attn_func` | Tiled KV loop, no T×T matrix, FA-2 parallelizes over queries | SDPA + causal mask + softmax + PV |
| FlashInfer | `single_decode_with_kv_cache` | Paged KV (non-contiguous blocks), GQA | decode attention with variable-length paged cache |
| FlashInfer | `single_prefill_with_kv_cache` | FA-style tiled prefill + paging | prefill attention |
| FlashInfer | Flash-Decoding | Splits K/V over P CTAs for long-context decode | reduces latency at T>2048 |
| vLLM | `paged_attention_v2.cu` | Block-table lookup, GQA, warp-level K/V head assignment | GQA + paged KV |
| SGLang | FlashInfer backend | RadixAttention: prefix sharing via radix tree | KV cache deduplication for common prefixes |
| TRT-LLM | FusedMHA plugin | QKV proj + attention + O proj in one TRT op | Full attention sublayer fused |

### SGLang RadixAttention

When multiple users share the same system prompt, SGLang stores the KV cache for that shared prefix **once** in a radix tree and reuses it for all users. This reduces KV computation and memory by:

$$\frac{\text{prefix length}}{\text{total context length}}$$

For a 2048-token system prompt shared across all users and total context of 4096 tokens, this is a **2× savings** in KV prefill compute and memory.

### vLLM PagedAttention

KV cache is divided into pages of block_size=16 tokens each. Pages are allocated on demand. They are **non-contiguous in memory** — the kernel uses block tables (arrays of GPU pointers) to find each page at runtime. This enables virtually unlimited context length within GPU memory, with no waste from padding to a fixed max length.

### decode vs. prefill split

At prefill (T tokens simultaneously): the score GEMM is Q[T,D_h] @ K[D_h,T] — a T×T matrix multiply, compute-bound at large T.

At decode (1 new token): the score GEMM is Q[1,D_h] @ K[D_h,T] — a GEMV (matrix-vector), memory-bound regardless of T.

This fundamental difference (GEMM vs. GEMV) is why FlashInfer exposes `single_decode` and `single_prefill` as separate kernels with different tuning rather than one unified kernel.

## Key Takeaways

- **What attention does:** For each query token, compute a weighted average of all other tokens' value vectors — where the weight measures how well that token's query matches each key. It's the only O(T²) operation in the transformer, and the only one where tokens "talk to each other" directly. Everything else (MLP, LayerNorm, projections) operates on one token at a time.

- **FlashAttention's key insight:** You never need the full T×T score matrix in memory. Three running scalars — m (running max), d (running denominator), acc (running output) — are enough to compute the exact same result in a single tile-by-tile pass through K and V. Those three scalars live in registers. The T×T matrix never exists.

- **Two fundamentally different regimes:** At decode (M=1 new token), the score computation Q·K^T is a matrix-vector multiply — memory-bound, bottlenecked by reading the KV cache from HBM. At prefill (M=T tokens), it's a full T×T GEMM — compute-bound above T≈600. This is not a tuning difference; it's a structural difference. Production systems (FlashInfer, vLLM) ship completely separate kernels for each regime.